In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, to_date
from pyspark.sql.types import DecimalType, IntegerType

# 1. Inicializando o motor e lendo a Camada Raw
spark = SparkSession.builder.appName("TCC_Silver_Definitiva").getOrCreate()
caminho_raw = "../data/raw/*.csv"
print("✅ Lendo arquivos CSV da Camada Raw...")
df_raw = spark.read.csv(caminho_raw, sep=";", header=True, encoding="utf-8")

df_silver = df_raw

# 2. Conversão da Data
print("⚙️ Convertendo data_base para DateType...")
df_silver = df_silver.withColumn("data_base", to_date(col("data_base"), "yyyy-MM-dd"))

# 3. Conversão de Inteiros
print("⚙️ Convertendo numero_de_operacoes para IntegerType...")
df_silver = df_silver.withColumn("numero_de_operacoes", col("numero_de_operacoes").cast(IntegerType()))

# 4. Conversão de Decimais
colunas_decimal = [
    "a_vencer_ate_90_dias", "a_vencer_de_91_ate_360_dias", "a_vencer_de_361_ate_1080_dias", 
    "a_vencer_de_1081_ate_1800_dias", "a_vencer_de_1801_ate_5400_dias", "a_vencer_acima_de_5400_dias", 
    "carteira_a_vencer", "vencido_de_15_ate_90_dias", "vencido_acima_de_90_dias", 
    "carteira_vencida", "carteira_ativa", "carteira_inadimplencia", "ativo_problematico"
]

print("⚙️ Convertendo colunas financeiras para DecimalType(18,2)...")
for c in colunas_decimal:
    df_silver = df_silver.withColumn(
        c, 
        regexp_replace(col(c), ",", ".").cast(DecimalType(18, 2))
    )

# 5. Sobrescrevendo a Camada Silver
caminho_parquet = "../data/processed/bacen_scr"
print("⏳ Gravando dados tipados em Parquet...")

df_silver.write \
    .mode("overwrite") \
    .parquet(caminho_parquet)

print(f"✅ Camada Silver selada e 100% tipada com sucesso em: {caminho_parquet}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/14 18:26:50 WARN Utils: Your hostname, DESKTOP-P0CSUOC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/14 18:26:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/haroldo/tcc_bacen_scr/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/14 18:26:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/14 18:26:53 WARN Utils: Service 'Spar

✅ Lendo arquivos CSV da Camada Raw...


26/10/14 18:26:57 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/raw/*.csv.
java.io.FileNotFoundException: File ../data/raw/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.analysis.ResolveData

⚙️ Convertendo data_base para DateType...
⚙️ Convertendo numero_de_operacoes para IntegerType...
⚙️ Convertendo colunas financeiras para DecimalType(18,2)...
⏳ Gravando dados tipados em Parquet...


26/10/14 18:27:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/10/14 18:27:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/10/14 18:27:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/10/14 18:27:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/10/14 18:27:42 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/10/14 18:27:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/10/14 18:27:43 WARN MemoryManager: Total allocation exceeds 95.00% 

✅ Camada Silver selada e 100% tipada com sucesso em: ../data/processed/bacen_scr


In [2]:
from pyspark.sql.functions import count, when, isnull

# 1. Estrutura e tipos das colunas
print("=== ESTRUTURA DOS DADOS (SCHEMA) ===")
df_silver.printSchema()

# 2. Primeiras 5 linhas
print("\n=== PRIMEIRAS 5 LINHAS ===")
df_silver.show(5, truncate=False)

# 3. Verificação de nulos por coluna
print("\n=== CONTAGEM DE VALORES NULOS ===")
df_silver.select([
    count(when(isnull(c), c)).alias(c) for c in df_silver.columns
]).show(vertical=True)

=== ESTRUTURA DOS DADOS (SCHEMA) ===
root
 |-- data_base: date (nullable = true)
 |-- uf: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- cliente: string (nullable = true)
 |-- cnae_ocupacao: string (nullable = true)
 |-- porte: string (nullable = true)
 |-- modalidade: string (nullable = true)
 |-- submodalidade: string (nullable = true)
 |-- origem: string (nullable = true)
 |-- indexador: string (nullable = true)
 |-- numero_de_operacoes: integer (nullable = true)
 |-- a_vencer_ate_90_dias: decimal(18,2) (nullable = true)
 |-- a_vencer_de_91_ate_360_dias: decimal(18,2) (nullable = true)
 |-- a_vencer_de_361_ate_1080_dias: decimal(18,2) (nullable = true)
 |-- a_vencer_de_1081_ate_1800_dias: decimal(18,2) (nullable = true)
 |-- a_vencer_de_1801_ate_5400_dias: decimal(18,2) (nullable = true)
 |-- a_vencer_acima_de_5400_dias: decimal(18,2) (nullable = true)
 |-- carteira_a_vencer: decimal(18,2) (nullable = true)
 |-- vencido_de_15_ate_90_dias: decimal(18,2) (nullab

-RECORD 0-----------------------------
 data_base                      | 0   
 uf                             | 0   
 segmento                       | 0   
 cliente                        | 0   
 cnae_ocupacao                  | 0   
 porte                          | 0   
 modalidade                     | 0   
 submodalidade                  | 0   
 origem                         | 0   
 indexador                      | 0   
 numero_de_operacoes            | 0   
 a_vencer_ate_90_dias           | 0   
 a_vencer_de_91_ate_360_dias    | 0   
 a_vencer_de_361_ate_1080_dias  | 0   
 a_vencer_de_1081_ate_1800_dias | 0   
 a_vencer_de_1801_ate_5400_dias | 0   
 a_vencer_acima_de_5400_dias    | 0   
 carteira_a_vencer              | 0   
 vencido_de_15_ate_90_dias      | 0   
 vencido_acima_de_90_dias       | 0   
 carteira_vencida               | 0   
 carteira_ativa                 | 0   
 carteira_inadimplencia         | 0   
 ativo_problematico             | 0   



In [3]:
from pyspark.sql.functions import countDistinct, col

print("=== AUDITORIA DE CARDINALIDADE (VALORES DISTINTOS) ===")

# Seleciona apenas as colunas que são texto (ignorando os números decimais)
colunas_qualitativas = [coluna for coluna, tipo in df_silver.dtypes if tipo == 'string']

# Monta o cálculo de contagem distinta para cada uma
expressoes_distintos = [
    countDistinct(col(c)).alias(c) for c in colunas_qualitativas
]

# Executa e exibe o resultado
df_distintos = df_silver.agg(*expressoes_distintos)
df_distintos.show(vertical=True)

=== AUDITORIA DE CARDINALIDADE (VALORES DISTINTOS) ===


-RECORD 0------------
 uf            | 27  
 segmento      | 8   
 cliente       | 2   
 cnae_ocupacao | 30  
 porte         | 13  
 modalidade    | 13  
 submodalidade | 56  
 origem        | 2   
 indexador     | 6   

